# NewsAPI Exploratory Data Collection

## Imports

In [91]:
import os
import time
import math
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from IPython.display import clear_output

## User Configuration

Edit the variables in this cell to control which companies are collected, the date window, the language and sort preferences, and where output files are written. The validation block at the bottom will catch common misconfigurations before any API calls are made.

In [92]:
# Date range
# Dates must be formatted as YYYY-MM-DD strings.
# The NewsAPI free tier only supports lookback windows of up to 30 days from today.
START_DATE = (datetime.now(timezone.utc) - timedelta(days=29)).strftime("%Y-%m-%d")
END_DATE   = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Company list
# Keys are the ticker symbols used for file naming and joining with stock data.
# Values are the search queries sent to NewsAPI - use specific names to reduce noise.
# GOOGL and GOOG intentionally share the same query string but are stored under
# separate tickers so each can be joined with its own stock price series.
COMPANIES = {
    "NVDA": "Nvidia",
    "AAPL": "Apple",
    "MSFT": "Microsoft",
    "AMZN": "Amazon",
    "GOOGL": "Alphabet Google",
    "AVGO": "Broadcom",
    "META": "Meta Platforms",
    "GOOG": "Alphabet Google",
    "TSLA": "Tesla",
    "BRK-B": "Berkshire Hathaway",
}

# Query options
# SORT_BY controls article ordering within each page: "publishedAt" returns the
# most recent articles first, "relevancy" ranks by closeness to the query string,
# and "popularity" surfaces articles from the most-trafficked sources first.
LANGUAGE = "en"
SORT_BY = "publishedAt"

# Volume limits
# PAGE_SIZE is the number of articles returned per request and cannot exceed 100.
# MAX_PAGES is the maximum number of pages fetched per monthly window; the free
# tier caps total results at 1000 per query, which is 10 full pages of 100.
PAGE_SIZE = 100
MAX_PAGES = 10

# Request throttle
# Seconds to sleep between successive API calls. Increase this value if you
# encounter rate-limit errors during bulk collection.
REQUEST_DELAY = 0.25

# Output paths
RAW_NEWS_DIR = "../raw_data/news"
PROCESSED_NEWS_DIR = "../processed_data/news"

os.makedirs(RAW_NEWS_DIR, exist_ok=True)
os.makedirs(PROCESSED_NEWS_DIR, exist_ok=True)

## Configuration

Sets up the API key and NewsAPI base endpoint. All request constants and company definitions are inherited from the User Configuration cell above.

In [93]:
NEWS_API_KEY = os.environ.get("NEWS_API_KEY")
if not NEWS_API_KEY:
    raise EnvironmentError("NEWS_API_KEY environment variable is not set.")

BASE_URL = "https://newsapi.org/v2/everything"

## Fetching News Articles

Defines helpers to fetch a single page of articles from NewsAPI, convert raw JSON into a DataFrame, split a date range into monthly windows, and paginate across all windows to collect the maximum number of articles for a given company.

In [87]:
# Sends a single request to the NewsAPI and returns the raw JSON response.
# Builds a parameter dictionary with the search query, date window, language filter, sort order, page size, and page number,
# then makes a request to the NewsAPI /everything endpoint.
def fetch_articles_page(query, from_date, to_date, page=1):
    params = {
        "q": query,
        "from": from_date,
        "to": to_date,
        "language": LANGUAGE,
        "sortBy": SORT_BY,
        "pageSize": PAGE_SIZE,
        "page": page,
        "apiKey": NEWS_API_KEY,
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=20)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request failed (page={page}, from={from_date}): {e}")
        return {"status": "error", "articles": [], "totalResults": 0}

# Converts a list of raw NewsAPI article dicts into a flat pandas DataFrame.
# Iterates over each article dict, extracts the relevant fields (published date, source, author, title, description, content, url), attaches the
# ticker label, and collects all rows into a DataFrame.
# The source field is a nested dict inside the article so it is unpacked separately before extraction.
def articles_to_dataframe(articles, ticker, query):
    if not articles:
        return pd.DataFrame()
    rows = []
    for art in articles:
        source = art.get("source") or {}
        rows.append({
            "ticker": ticker,
            "published_at": art.get("publishedAt"),
            "source_name": source.get("name"),
            "author": art.get("author"),
            "title": art.get("title"),
            "description": art.get("description"),
            "content": art.get("content"), # truncated at 200 chars on free tier
            "url": art.get("url"),
        })
    return pd.DataFrame(rows)

# Splits a date range into a list of monthly from/to pairs.
# Parses the start and end dates as datetime objects then walks forward one month at a time, computing the last day of each calendar month and
# clamping it to the overall end date. Each iteration appends a (from, to) string tuple to the list.
def build_monthly_date_ranges(start_date, end_date):
    ranges = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    while current <= end:
        if current.month == 12:
            month_end = current.replace(day=31)
        else:
            month_end = current.replace(month=current.month + 1, day=1) - timedelta(days=1)
        period_end = min(month_end, end)
        ranges.append((current.strftime("%Y-%m-%d"), period_end.strftime("%Y-%m-%d")))
        current = period_end + timedelta(days=1)
    return ranges

# Collects all available articles for a company across the full date range.
# Calls build_monthly_date_ranges to split the window into months, then for each month it paginates through up to MAX_PAGES pages using
# fetch_articles_page, stopping early if the API signals no more results or returns
# an error. A short sleep is added between requests to avoid rate limiting.
# All collected articles are converted to DataFrames via articles_to_dataframe and concatenated into a single DataFrame at the end.
def fetch_all_articles_for_company(ticker, query, start_date, end_date):
    date_ranges = build_monthly_date_ranges(start_date, end_date)
    all_frames = []

    for from_dt, to_dt in date_ranges:
        print(f"  [{ticker}] Fetching {from_dt} to {to_dt} ...", end=" ")
        period_articles = []

        for page in range(1, MAX_PAGES + 1):
            data = fetch_articles_page(query, from_dt, to_dt, page=page)
            articles = data.get("articles", [])
            period_articles.extend(articles)

            total_results = data.get("totalResults", 0)
            pages_needed = min(math.ceil(total_results / PAGE_SIZE), MAX_PAGES)
            time.sleep(REQUEST_DELAY)

            if data.get("status") != "ok" or page >= pages_needed:
                break

        print(f"{len(period_articles)} articles")
        df = articles_to_dataframe(period_articles, ticker, query)
        if not df.empty:
            all_frames.append(df)

    if not all_frames:
        return pd.DataFrame()
    return pd.concat(all_frames, ignore_index=True)

## Data Cleaning

Parses timestamps to UTC, adds a date column for joining with stock data, drops rows with no usable text or NewsAPI placeholder articles, deduplicates on URL and ticker, and strips whitespace from all text fields.

In [94]:
TEXT_COLUMNS = ["title", "description", "content", "author", "source_name"]

# Cleans a raw NewsAPI DataFrame for sentiment analysis.
# Parses the published_at column to timezone-aware UTC datetimes and derives a normalized date column for stock data joins.
# It then drops rows where both title and description are null, removes rows where the title is the NewsAPI placeholder string
# [Removed], and deduplicates on url and ticker to prevent the same article from
# appearing multiple times across paginated results. Finally it strips whitespace from all text columns and replaces empty or nan strings with proper NA values
# before sorting by published_at and resetting the index.
def clean_news_df(df):
    if df.empty:
        return df
    df = df.copy()

    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["date"] = df["published_at"].dt.normalize() # date only, used for joining with stock data

    df.dropna(subset=["title", "description"], how="all", inplace=True)
    df = df[~df["title"].str.strip().eq("[Removed]")] # drop NewsAPI placeholder articles
    df.drop_duplicates(subset=["url", "ticker"], keep="first", inplace=True)

    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace({"nan": pd.NA, "": pd.NA})

    df.sort_values("published_at", ascending=True, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.index.name = "article_id"
    return df


# Prints a descriptive summary of a cleaned news DataFrame for a given ticker.
# Reports row count, date range, number of unique sources, null counts for the three main text fields, mean articles per day computed by grouping
# on the date column, and the top 5 sources by article count.
def describe_news_df(df, ticker):
    print(f"\n{'='*50}")
    print(f"Ticker : {ticker} | Rows : {len(df):,}")
    if df.empty:
        return
    print(f"Date range: {df['published_at'].min().date()} to {df['published_at'].max().date()}")
    print(f"Unique sources: {df['source_name'].nunique()}")
    print(f"Null title: {df['title'].isna().sum()}")
    print(f"Null description: {df['description'].isna().sum()}")
    print(f"Null content: {df['content'].isna().sum()}")
    print(f"Mean articles/day: {df.groupby('date').size().mean():.1f}")
    print(f"\nTop 5 sources:")
    for src, cnt in df["source_name"].value_counts().head(5).items():
        print(f"  {src:<35} {cnt}")
    print(f"{'='*50}")

## Single Company Example (TSLA)

Fetches and cleans news articles for TSLA. Run this first to verify your API key and inspect the shape of the data before running bulk collection.

In [95]:
tsla_raw = fetch_all_articles_for_company("TSLA", COMPANIES["TSLA"], START_DATE, END_DATE)
tsla_raw.to_csv(f"{RAW_NEWS_DIR}/TSLA_news_raw.csv", index=False)

tsla_clean = clean_news_df(tsla_raw)
tsla_clean.to_csv(f"{PROCESSED_NEWS_DIR}/TSLA_news_clean.csv", index=True)
describe_news_df(tsla_clean, "TSLA")

tsla_clean[["published_at", "date", "source_name", "title", "description", "content"]].head(10)

  [TSLA] Fetching 2026-03-22 to 2026-03-31 ... Request failed (page=1, from=2026-03-22): 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=Tesla&from=2026-03-22&to=2026-03-31&language=en&sortBy=publishedAt&pageSize=100&page=1&apiKey=e8d5066988f74fb1bba8c5a2350fe47b
0 articles
  [TSLA] Fetching 2026-04-01 to 2026-04-20 ... Request failed (page=1, from=2026-04-01): 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=Tesla&from=2026-04-01&to=2026-04-20&language=en&sortBy=publishedAt&pageSize=100&page=1&apiKey=e8d5066988f74fb1bba8c5a2350fe47b
0 articles

Ticker : TSLA | Rows : 0


KeyError: "None of [Index(['published_at', 'date', 'source_name', 'title', 'description',\n       'content'],\n      dtype='str')] are in the [columns]"

## Bulk Collection for All Companies

Loops over every company in COMPANIES, fetches and cleans articles, saves a raw and cleaned CSV per ticker, then concatenates everything into a single master CSV ready for sentiment analysis.

In [ ]:
# GOOGL and GOOG share the same query string but are stored under separate tickers
# so they can each be joined with their respective stock price series

all_clean_frames = []
total_companies = len(COMPANIES)

for idx, (ticker, query) in enumerate(COMPANIES.items(), start=1):
    clear_output(wait=True)
    print(f"[{idx}/{total_companies}] Collecting news for {ticker} ('{query}') ...\n")

    raw_df = fetch_all_articles_for_company(ticker, query, START_DATE, END_DATE)

    if raw_df.empty:
        print(f"  No articles returned for {ticker}. Skipping.")
        continue

    raw_df.to_csv(f"{RAW_NEWS_DIR}/{ticker}_news_raw.csv", index=False)

    clean_df = clean_news_df(raw_df)
    clean_df.to_csv(f"{PROCESSED_NEWS_DIR}/{ticker}_news_clean.csv", index=True)
    describe_news_df(clean_df, ticker)

    all_clean_frames.append(clean_df)

print("\nBulk collection complete.")

[10/10] Collecting news for BRK-B ('Berkshire Hathaway') ...

  [BRK-B] Fetching 2026-03-22 to 2026-03-31 ... Request failed (page=1, from=2026-03-22): 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=Berkshire+Hathaway&from=2026-03-22&to=2026-03-31&language=en&sortBy=publishedAt&pageSize=100&page=1&apiKey=e8d5066988f74fb1bba8c5a2350fe47b
0 articles
  [BRK-B] Fetching 2026-04-01 to 2026-04-20 ... Request failed (page=1, from=2026-04-01): 429 Client Error: Too Many Requests for url: https://newsapi.org/v2/everything?q=Berkshire+Hathaway&from=2026-04-01&to=2026-04-20&language=en&sortBy=publishedAt&pageSize=100&page=1&apiKey=e8d5066988f74fb1bba8c5a2350fe47b
0 articles
  No articles returned for BRK-B. Skipping.

Bulk collection complete.


In [ ]:
if all_clean_frames:
    master_df = pd.concat(all_clean_frames, ignore_index=True)
    master_df.index.name = "article_id"
    master_df.to_csv(f"{PROCESSED_NEWS_DIR}/all_companies_news_clean.csv", index=True)

    print(f"Master DataFrame shape: {master_df.shape}")
    print(f"\nArticles per ticker:")
    print(master_df["ticker"].value_counts().to_string())
else:
    print("No data collected.")

Master DataFrame shape: (1456, 9)

Articles per ticker:
ticker
AMZN     199
NVDA     198
MSFT     196
GOOGL    195
GOOG     195
AAPL     193
TSLA     191
AVGO      89
